# Configuring & Running the EF5 PET/CT + ROI Slice Generator (`ef5_slice_cli`)

This mini-notebook demonstrates how to configure and run `src.ef5_slice_cli` to generate scrollable, pixel-aligned **CT**, **PET**, and **PET/CT Overlay** slices (with superimposed RT_Image ROIs) across all 3 orthogonal axes (`transaxial`, `coronal`, `sagittal`) at any slice spacing — from coarse overview steps all the way down to **every individual PET slice (`step_pet_slices=1`) and its closest equivalent CT slice**.

It works identically on:
- **Set 1 (`c1_M0`)**: `images/set1/{c1_M0_PET_mouse.dcm, c1_M0_CT_mouse.dcm, rois.ROI}`
- **Set 2 Cropped Mice (`c8_M0`, `c8_M1R`, `c8_M1L`, `c8_M1L1R`)**: `outputs/set2/cropped_mice/<mouse_id>/{<mouse_id>_PET_mouse.dcm, <mouse_id>_CT_mouse.dcm, rois.ROI}`
- **Set 2 Full-Bed Volumes**: `images/set2/{day2_set5_pet.dcm, i100118_v1.ct.img, mice_tumor_rois.ROI}`

## Output Folder Structure
For each run, `ef5_slice_cli` creates:
- `roi_summary_stats.csv` & `roi_summary_stats.txt`: Quantitative ROI statistics (`Points`, `Volume`, `Min`, `Max`, `Mean`, `Std Dev`, `Sum`, `TBR_mean`, `TBR_max`).
- `slice_manifest.csv`: A lookup table mapping every generated frame to its exact `pet_slice_index`, `pet_slice_coord_mm`, `closest_ct_slice_index`, `ct_slice_coord_mm`, and `rois_present`.
- `scrollable_by_slice/<plane>/`: Ordered `*_1_CT.png`, `*_2_PET.png`, `*_3_Overlay.png` per slice so you can flip back and forth between modalities at the exact same slice.
- `scrollable_by_modality/<plane>/{CT,PET,Overlay}/`: All slices for a single modality in sequential order so you can scroll smoothly through the 3D volume from end to end.
- `panels/<plane>/`: Side-by-side 3-panel comparison images (`CT | PET | Overlay`).

## Option 1: Configure and Run via Python (`SliceExportConfig`)

In [ ]:
import sys
from pathlib import Path

root = Path('/workspace') if Path('/workspace').exists() else Path('..').resolve()
if str(root) not in sys.path:
    sys.path.insert(0, str(root))

from src.ef5_slice_cli import SliceExportConfig, run_slice_generator

# Choose dataset folder:
# - Set 1: root / 'images/set1'
# - Set 2 cropped mouse (e.g. c8_M0, c8_M1R, c8_M1L, c8_M1L1R): root / 'outputs/set2/cropped_mice/c8_M0'
config = SliceExportConfig(
    pet_path=root / 'images/set1/c1_M0_PET_mouse.dcm',
    ct_path=root / 'images/set1/c1_M0_CT_mouse.dcm',
    roi_path=root / 'images/set1/rois.ROI',
    out_dir=root / 'outputs/set1_every_pet_slice',
    planes=('transaxial', 'coronal', 'sagittal'),
    step_pet_slices=1,          # 1 = every single PET slice + closest equivalent CT slice
    step_mm=None,               # Or set e.g. step_mm=1.6 to step by physical millimeters
    scope='roi',                # 'roi' (slices intersecting ROIs) or 'all' (entire volume)
    margin_mm=0.0,              # Extra padding in mm around the ROI bounding box
    modes=('CT', 'PET', 'Overlay', 'Panel'),
    ct_clim=(-500.0, 800.0),    # CT window (HU)
    pet_clim=(0.0, 2.2),        # EF5 PET window (%IDG)
    overlay_alpha=0.78,         # Max opacity of PET overlay on CT
    crop_long_axis_to_rois=True,# Zoom coronal/sagittal Y-axis around tumour region
    long_axis_pad_mm=6.0,
    dpi=120,
)

stats_df, manifest_df = run_slice_generator(config)
display(stats_df)
display(manifest_df.head(15))

## Option 2: Run on Any Set 2 Cropped Mouse (`c8_M0`, `c8_M1R`, `c8_M1L`, `c8_M1L1R`)

In [ ]:
mouse_id = 'c8_M1L'  # Choose from: 'c8_M0', 'c8_M1R', 'c8_M1L', 'c8_M1L1R'
mouse_dir = root / 'outputs/set2/cropped_mice' / mouse_id

cfg_set2 = SliceExportConfig(
    pet_path=mouse_dir / f'{mouse_id}_PET_mouse.dcm',
    ct_path=mouse_dir / f'{mouse_id}_CT_mouse.dcm',
    roi_path=mouse_dir / 'rois.ROI',
    out_dir=root / 'outputs/set2/mice_slices_every_pet_slice' / mouse_id,
    step_pet_slices=1,          # Every single PET slice + closest CT slice
    scope='roi',
    pet_clim=(0.0, 2.8),
)

stats_m, manifest_m = run_slice_generator(cfg_set2)
display(stats_m)

## Option 3: Run via Command Line (`python -m src.ef5_slice_cli`)

In [ ]:
!python -m src.ef5_slice_cli --help

In [ ]:
# Example: Generate every single PET slice (step=1) for Set 2 mouse c8_M0
!python -m src.ef5_slice_cli \
    --pet /workspace/outputs/set2/cropped_mice/c8_M0/c8_M0_PET_mouse.dcm \
    --ct /workspace/outputs/set2/cropped_mice/c8_M0/c8_M0_CT_mouse.dcm \
    --roi /workspace/outputs/set2/cropped_mice/c8_M0/rois.ROI \
    --out-dir /workspace/outputs/set2/c8_M0_step1 \
    --step-pet-slices 1 \
    --pet-clim 0.0 2.8 \
    --scope roi